In [2]:
import cv2
import os
import math
import pandas as pd

# ==========================================
# 파라미터 세팅
# ==========================================
CSV_FILE_PATH = "shorts-analysis_test-1.csv"
VIDEO_DIR = "data"
FRAME_DIR = "data/frames_for_vlm" # VLM 전용 폴더

# 💡 모델의 메모리 한계를 고려해 1초에 1장만 뽑습니다. (필요시 2로 변경)
TARGET_FPS = 1 

os.makedirs(FRAME_DIR, exist_ok=True)

# ==========================================
# 프레임 추출 로직
# ==========================================
try:
    df = pd.read_csv(CSV_FILE_PATH)
except Exception as e:
    print(f"❌ CSV 로드 에러: {e}")
    df = pd.DataFrame()

success_count = 0

for index, row in df.iterrows():
    # 1. URL 컬럼 가져오기
    vid_url = row['final_url']
    
    # 2. URL에서 실제 유튜브 비디오 ID만 깔끔하게 추출 (문자열 슬라이싱)
    # 예: https://www.youtube.com/shorts/mfi0n3oNABM -> mfi0n3oNABM
    real_vid_id = vid_url.split('/')[-1].split('?')[0]
    
    # 3. 실제 ID로 경로 생성
    video_path = f"{VIDEO_DIR}/{real_vid_id}.mp4"
    
    # 기존 코드 로직 유지 (폴더명도 실제 ID로 생성)
    vid_id = real_vid_id
    
    print(f"\n🎬 [{index+1}/{len(df)}] VLM 데이터 전처리 시작: {vid_id}")
    
    if not os.path.exists(video_path):
        print(f" ⚠️ 영상이 없습니다. 스킵: {video_path}")
        continue
        
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(" ❌ 영상을 읽을 수 없습니다.")
        continue

    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps == 0:
        continue
        
    # 원본 FPS 대비 몇 프레임마다 한 번씩 저장할지 계산 (예: 25fps 영상에서 1fps 추출하려면 25장마다 저장)
    frame_interval = math.ceil(original_fps / TARGET_FPS)
    
    # 비디오 ID별로 프레임을 담을 방 생성
    out_folder = f"{FRAME_DIR}/{vid_id}"
    os.makedirs(out_folder, exist_ok=True)

    frame_count = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 지정된 간격마다 프레임 저장
        if frame_count % frame_interval == 0:
            # VLM이 순서를 알 수 있도록 001, 002 형식으로 저장
            save_path = f"{out_folder}/frame_{saved_count:03d}.jpg"
            cv2.imwrite(save_path, frame)
            saved_count += 1

        frame_count += 1

    cap.release()
    print(f" ✅ 추출 완료! 총 {saved_count}장의 이미지가 준비되었습니다.")
    success_count += 1

print(f"\n🎉 VLM용 데이터 전처리 완료! (성공: {success_count}개 영상)")


🎬 [1/6] VLM 데이터 전처리 시작: knCQBlBKSRM
 ✅ 추출 완료! 총 21장의 이미지가 준비되었습니다.

🎬 [2/6] VLM 데이터 전처리 시작: NCVxpXxTMSU
 ✅ 추출 완료! 총 12장의 이미지가 준비되었습니다.

🎬 [3/6] VLM 데이터 전처리 시작: LRmLOsmMHts
 ✅ 추출 완료! 총 30장의 이미지가 준비되었습니다.

🎬 [4/6] VLM 데이터 전처리 시작: twi9zUxsXu0
 ✅ 추출 완료! 총 55장의 이미지가 준비되었습니다.

🎬 [5/6] VLM 데이터 전처리 시작: Xfu1kCID0Ls
 ✅ 추출 완료! 총 23장의 이미지가 준비되었습니다.

🎬 [6/6] VLM 데이터 전처리 시작: MIJR3Dm0YOk
 ✅ 추출 완료! 총 50장의 이미지가 준비되었습니다.

🎉 VLM용 데이터 전처리 완료! (성공: 6개 영상)
